# Exploration trainset

In [1]:
import numpy as np 
import cbor2 
from pathlib import Path
import pandas as pd 

base = Path.cwd() 
target = base / "../../../unprocessedAllButBenchmark.v2.1/"

!tree -L 1 {target}

/tempory/the_three_potatoes/ri_project/workspaces/amelie/RI_Project/Notebooks/../../../unprocessedAllButBenchmark.v2.1
├── fold-0-unprocessedAllButBenchmark.Y2.cbor
├── fold-1-unprocessedAllButBenchmark.Y2.cbor
├── fold-2-unprocessedAllButBenchmark.Y2.cbor
├── fold-3-unprocessedAllButBenchmark.Y2.cbor
├── fold-4-unprocessedAllButBenchmark.Y2.cbor
├── LICENSE -> ../../enwiki-20161220/release-v2.0.1/unprocessedAllButBenchmark.cbor/LICENSE
├── README.mkd -> ../../enwiki-20161220/release-v2.0.1/unprocessedAllButBenchmark.cbor/README.mkd
└── unprocessedAllButBenchmark.Y2.cbor

1 directory, 8 files


In [2]:
path = target / "fold-0-unprocessedAllButBenchmark.Y2.cbor"

In [3]:
# data= []

# with open(path, "rb") as f:
#     for i in range(2): # only 2 objects !
#         obj = cbor2.load(f)
#         data.append(obj)
#         print(type(obj))
#         print("=" * 50)

In [4]:
from trec_car.read_data import iter_annotations

data = []

with open(path, "rb") as f:
    for i, page in enumerate(iter_annotations(f)):
        if i > 1:   # stop early (≈ sample)
            break
        print("name", page.page_name)
        print("id",page.page_id)
        print("type", page.page_type)
        print("meta", page.page_meta)
        print("meta type", type(page.page_meta))
        print("skeleton", page.skeleton)
        print("skeleton length", len(page.skeleton))
    
        for section_path in page.flat_headings_list():
            
            headings = [s.heading for s in section_path]
            print("headings length", len(headings))
            print(" / ".join(headings))
        

name Altruism
id enwiki:Altruism
type ArticlePage
meta  redirected = Unselfishness, Altutrists, Altruist, Pathological altruism, Altruistic, Altruistically, Problem of love, Digital altruism, The problem of love, Otherism, Altruists, Truist, Altruistic behavior, Altruistical, Altruism (philosophy), Selflessly, Alturism, Selfless action, Law of mutual aid 
 disambiguated = Benevolence, Selfless, Selflessness, Egoism 
 categories = Category:Altruism, Category:Auguste Comte, Category:Defence mechanisms, Category:Evolutionary psychology, Category:Morality, Category:Philanthropy, Category:Social philosophy, Category:Interpersonal relationships, Category:Virtue 
 inlinks = enwiki:Binding%20of%20Isaac, enwiki:Parent%E2%80%93offspring%20conflict, enwiki:Hades, enwiki:East%20Asian%20cultural%20sphere, enwiki:Collaborative%20partnerships, enwiki:Sigma%20Pi, enwiki:Mark%20Snyder%20(psychologist), enwiki:Asceticism, enwiki:Batgirl, enwiki:Prosocial%20behavior, enwiki:Xylocopa%20sulcatipes, enwiki:

In [ ]:
from trec_car.read_data import iter_annotations, Para, Section, List, Image

def read_train(path, nb_max = 10):
    df = []
    nb = 0
    with open(path, 'rb') as f:
        for page in iter_annotations(f):
            nb += 1
            if nb > nb_max:
                break
            for section_path in page.flat_headings_list():
                headings = [page.page_name] + [s.heading for s in section_path]
                # print(" / ".join(headings))
                leaf = section_path[-1]
                
                def extract_paras(children):
                    rows = []
                    for child in children:
                        if isinstance(child, Para):
                            rows.append([
                                page.page_id, page.page_name, ' / '.join(headings),
                                child.paragraph.para_id, child.paragraph.get_text(), "paragraph"
                            ])
                        elif isinstance(child, Section):
                            rows.extend(extract_paras(child.children))
                        elif isinstance(child, List):
                            rows.append([
                                page.page_id, page.page_name, ' / '.join(headings),
                                child.body.para_id, child.body.get_text(), "list"
                            ])
                        elif isinstance(child, Image):
                            pass
                        else:
                            print("Unknown child type:", type(child))
                            
                    return rows
                
                df.extend(extract_paras(leaf.children))

    df = pd.DataFrame(df, columns=['page_id', 'page_name', 'headings', 'para_id', 'text', 'type'])
    print("nb of pages:", nb)
    return df

In [16]:
train_data = read_train(path)
train_data.head()

nb of pages: 11


,page_id,page_name,headings,para_id,text,type
0,enwiki:Altruism,Altruism,Altruism / The notion of altruism,270b25b5d883d892ee9b786caa4023aea89c9c1e,The concept has a long history in philosophica...,paragraph
1,enwiki:Altruism,Altruism,Altruism / Scientific viewpoints,a0b1ab9826b606549ac92dae3e0401f0a4529f9b,Marcel Mauss's book The Gift contains a passag...,paragraph
2,enwiki:Altruism,Altruism,Altruism / Scientific viewpoints,bc5df4178b87672b3d05a153a099d0b598030d49,Compare Altruism (ethics) – perception of altr...,list
3,enwiki:Altruism,Altruism,Altruism / Scientific viewpoints,f77f84ff55ecafa9cef3a5a6368d26564b910188,Compare explanation of alms in various scriptu...,list
4,enwiki:Altruism,Altruism,Altruism / Scientific viewpoints,c0a614c6cd56067450c9e05a8ebba62fb35078e3,In the science of ethology (the study of anima...,paragraph


In [19]:
train_data.iloc[0].text

'The concept has a long history in philosophical and ethical thought. The term was originally coined in the 19th century by the founding sociologist and philosopher of science, Auguste Comte, and has become a major topic for psychologists (especially evolutionary psychology researchers), evolutionary biologists, and ethologists. Whilst ideas about altruism from one field can affect the other fields, the different methods and focuses of these fields always lead to different perspectives on altruism. In simple terms, altruism is caring about the welfare of other people and acting to help them.'

In [20]:
train_data.type.unique()

<StringArray>
['paragraph', 'list']
Length: 2, dtype: str